In [3]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("Downloads/Nifty 50 Historical Data.csv")

# Clean columns
df.columns = [c.strip() for c in df.columns]

# Date format
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Convert price column
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace(',', '')
    .astype(float)
)

# ----------------------
# Bollinger Bands
# ----------------------
window = 20

df['MA20'] = df['Price'].rolling(window).mean()
df['STD20'] = df['Price'].rolling(window).std()

df['UpperBand'] = df['MA20'] + 2 * df['STD20']
df['LowerBand'] = df['MA20'] - 2 * df['STD20']

# ----------------------
# Z-score
# ----------------------
df['Zscore'] = (df['Price'] - df['MA20']) / df['STD20']

# ----------------------
# RSI
# ----------------------
delta = df['Price'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df['RSI'] = 100 - (100/(1+rs))

# ----------------------
# Generate positions
# ----------------------
position = 0
entry_price = 0

positions = []

stop_loss = 0.03
take_profit = 0.06

for i in range(len(df)):

    price = df.loc[i, 'Price']

    if position == 0:

        # LONG
        if (
            price <= df.loc[i, 'LowerBand']
            and df.loc[i, 'Zscore'] < -1.5
            and df.loc[i, 'RSI'] < 30
        ):
            position = 1
            entry_price = price

        # SHORT
        elif (
            price >= df.loc[i, 'UpperBand']
            and df.loc[i, 'Zscore'] > 1.5
            and df.loc[i, 'RSI'] > 70
        ):
            position = -1
            entry_price = price

    elif position == 1:

        ret = (price - entry_price)/entry_price

        if (
            ret <= -stop_loss
            or ret >= take_profit
            or price >= df.loc[i, 'MA20']
        ):
            position = 0

    elif position == -1:

        ret = (entry_price - price)/entry_price

        if (
            ret <= -stop_loss
            or ret >= take_profit
            or price <= df.loc[i, 'MA20']
        ):
            position = 0

    positions.append(position)

# Store result
result = pd.DataFrame({
    'Date': df['Date'],
    'Position': positions
})

result.to_csv("strategy_output.csv", index=False)

print(result.head())

        Date  Position
0 2026-01-01         0
1 2025-12-31         0
2 2025-12-30         0
3 2025-12-29         0
4 2025-12-26         0
